# Toolkits 16 aula

Toolkits são conjuntos de ferramentas. Para alguns toolkits, agent específicos já foram criado na estrutura do LangChain e é isto que vamos explorar agora.

https://python.langchain.com/v0.1/docs/integrations/toolkits/

## Pandas Dataframe

In [5]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI
import pandas as pd

df = pd.read_csv(
    "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv"
)
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [6]:
df.shape

(891, 12)

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")
agent = create_pandas_dataframe_agent(
    chat,
    df,
    verbose=True,
    agent_type='tool-calling',
    allow_dangerous_code=True #Adicionando método necessário na versão 0.3 do langchain
)

In [8]:
agent.invoke({'input': 'Quantas linhas tem na tabela?'})



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'print(len(df))'}`


891
A tabela tem 891 linhas.

> Finished chain.


{'input': 'Quantas linhas tem na tabela?',
 'output': 'A tabela tem 891 linhas.'}

In [9]:
agent.invoke({'input': 'Qual a média da idade dos passageiros?'})



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "print(df['Age'].mean())"}`


29.69911764705882
A idade média dos passageiros é 29.699.


> Finished chain.


{'input': 'Qual a média da idade dos passageiros?',
 'output': 'A idade média dos passageiros é 29.699.\n'}

In [10]:
agent.invoke({'input': 'Quantos passageiros sobreviveram?'})



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "print(df['Survived'].sum())"}`


342
342 passageiros sobreviveram.

> Finished chain.


{'input': 'Quantos passageiros sobreviveram?',
 'output': '342 passageiros sobreviveram.'}

## SQL Database

In [14]:
from langchain_community.utilities.sql_database import SQLDatabase

db = SQLDatabase.from_uri('sqlite:///arquivos/Chinook.db')

In [15]:
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_google_genai import ChatGoogleGenerativeAI

chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")

agent_executor = create_sql_agent(
    chat,
    db=db,
    agent_type='tool-calling',
    verbose=True
)

In [16]:
for tool in agent_executor.tools:
    print(tool.name)
    print(tool.description)
    print()

sql_db_query
Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema
Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables
Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker
Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [18]:
agent_executor.invoke({'input': 'Me descreva a base de dados'})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`
responded: 


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Invoking: `sql_db_schema` with `{'table_names': 'Album, Artist, Track'}`
responded: Now that I know the tables, I can describe them. I'll start with a few of the most important ones.



CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Track" (
	"TrackId" INTEGER NOT NULL, 
	"Name" NVARCHAR(200)

{'input': 'Me descreva a base de dados',
 'output': "The database contains information about music. There are tables for Albums, Artists, Tracks, Genres, Media Types, Playlists, Customers, Employees, Invoices, and Invoice Lines. The Album table contains the album's title and the artist's ID. The Artist table contains the artist's name. The Track table contains the track's name, album ID, media type ID, genre ID, composer, length in milliseconds, size in bytes, and unit price.\n"}

In [19]:
agent_executor.invoke({'input': 'Qual artista possui mais albuns?'})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`
responded: 


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Invoking: `sql_db_schema` with `{'table_names': 'Album,Artist'}`
responded: I see the tables `Album` and `Artist`. I will query the schema for these tables.



CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT Artist.Name, COUNT(Album.AlbumId) FROM Artist

{'input': 'Qual artista possui mais albuns?',
 'output': 'Iron Maiden tem mais albuns.'}

In [20]:
agent_executor.invoke({'input': 'Qual o cliente que mais comprou?'})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`
responded: 


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
Invoking: `sql_db_schema` with `{'table_names': 'Customer,Invoice'}`
responded: I should look at the schema of the `Customer` and `Invoice` tables since the question is about customers and purchases.



CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Customer table:
CustomerId	FirstName	LastName	Company	Address	City	State	Country	Post

{'input': 'Qual o cliente que mais comprou?',
 'output': 'O cliente que mais comprou foi Helena Holý.'}